In [1]:
try:
    import torch_geometric
except ImportError:
    import subprocess
    print("Installing torch_geometric on Kaggle environment...")
    subprocess.run(["pip", "install", "torch-geometric", "-q"])
    import torch_geometric
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

PyTorch Geometric version: 2.8.0.post1


# Temporal Data Splitting

Takes the five PyG tensors from graph_feature_construction and splits them
chronologically into Train (60%), Validation (20%), and Test (20%) sets.

Why temporal splitting and not random?
Because randomly picking rows from Day 50 for training and Day 10 for testing
would mean the model is trained on future data and tested on past data (data leakage).
Temporal splitting ensures the model only learns from past transactions and
is evaluated on truly unseen future transactions.

1. Load the PyG graph_data Object from graph_feature_construction

In [2]:
import torch
import os

paths = [
    "/home/shreyas-nalle/Desktop/Delusional/model/graph_data.pt",
    "graph_data.pt",
    "/kaggle/working/graph_data.pt"
]
graph_path = next((p for p in paths if os.path.exists(p)), None)
if graph_path is None:
    raise FileNotFoundError("graph_data.pt not found locally! Run graph_feature_construction.ipynb first.")

graph_data = torch.load(graph_path, weights_only=False)
x = graph_data["x"]
edge_index = graph_data["edge_index"]
edge_attr = graph_data["edge_attr"]
timestamps = graph_data["timestamps"]
y = graph_data["y"]

print(f"Loaded graph data from {graph_path}:")
print(f"  x          : {x.shape}")
print(f"  edge_index : {edge_index.shape}")
print(f"  edge_attr  : {edge_attr.shape}")
print(f"  timestamps : {timestamps.shape}")
print(f"  y          : {y.shape} (Illicit ratio: {y.float().mean()*100:.3f}%)")

FileNotFoundError: [Errno 2] No such file or directory: 'model/graph_data.pt'

2. Bucket Transactions into Calendar Days

For every calendar day from Day 0 to Day n_days, we find which transaction
indices fall within that day's time window (l to r seconds).
We also track the illicit ratio per day for diagnostic purposes.

In [5]:
import numpy as np
import itertools

n_days = int(timestamps.max() / (3600 * 24) + 1)
daily_irs = []
daily_inds = []
daily_trans = []

for day in range(n_days):
    l = day * 24 * 3600
    r = (day + 1) * 24 * 3600
    day_inds = torch.where((timestamps >= l) & (timestamps < r))[0]
    daily_irs.append(y[day_inds].float().mean().item())
    daily_inds.append(day_inds)
    daily_trans.append(day_inds.shape[0])

print(f"Days processed : {n_days}")
print(f"Transactions per day (first 5 days) : {daily_trans[:5]}")
print(f"Illicit ratio per day (first 5 days) : {[round(r, 4) for r in daily_irs[:5]]}")

3. Find the Optimal Train / Validation / Test Day Split Points

The repo tries every combination of two day-index cut-points (i, j) using
itertools.combinations and picks the pair that produces split proportions
closest to [60%, 20%, 20%].
The score for each (i, j) pair is the maximum relative error across the three splits.
The pair with the lowest score wins.

In [6]:
split_per = [0.6, 0.2, 0.2]
daily_totals = np.array(daily_trans)
d_ts = daily_totals
I = list(range(len(d_ts)))
split_scores = {}

for i, j in itertools.combinations(I, 2):
    if j >= i:
        split_totals = [d_ts[:i].sum(), d_ts[i:j].sum(), d_ts[j:].sum()]
        split_totals_sum = np.sum(split_totals)
        split_props = [v / split_totals_sum for v in split_totals]
        split_error = [abs(v - t) / t for v, t in zip(split_props, split_per)]
        score = max(split_error)
        split_scores[(i, j)] = score

best_i, best_j = min(split_scores, key=split_scores.get)

split = [
    list(range(best_i)), 
    list(range(best_i, best_j)),                
    list(range(best_j, len(daily_totals)))     
]

print(f'Optimal cut-points : Day {best_i} and Day {best_j}')
print(f'Train days : Day 0 to Day {best_i - 1}  ({len(split[0])} days)')
print(f'Validation days : Day {best_i} to Day {best_j - 1}  ({len(split[1])} days)')
print(f'Test days : Day {best_j} to Day {n_days - 1}  ({len(split[2])} days)')

4. Collect Transaction Indices for Each Split

For each split (train, val, test), collect the edge indices from all days
belonging to that split and concatenate them into a single index tensor.

In [7]:
split_inds = {k : [] for k in range(3)}
for k in range(3):
    for day in split[k]:
        split_inds[k].append(daily_inds[day])

tr_inds = torch.cat(split_inds[0])
val_inds = torch.cat(split_inds[1])
te_inds = torch.cat(split_inds[2])

total = y.shape[0]
print(f'Train transactions: {tr_inds.shape[0]} ({tr_inds.shape[0]/total*100:.1f}%) || Illicit : {y[tr_inds].float().mean()*100:.3f}%')
print(f'Val transactions: {val_inds.shape[0]} ({val_inds.shape[0]/total*100:.1f}%) || Illicit : {y[val_inds].float().mean()*100:.3f}%')
print(f'Test transactions: {te_inds.shape[0]} ({te_inds.shape[0]/total*100:.1f}%) || Illicit : {y[te_inds].float().mean()*100:.3f}%')

5. Create Train, Validation and Test PyG Data Objects

Important notes from the repo:
- Train data: only edges from train days
- Validation data: edges from train + val days (cumulative history so the model knows past context)
- Test data: ALL edges (full graph history available at test time)
All three splits share the same full node feature matrix x.

In [9]:
from torch_geometric.data import Data
tr_x = val_x = te_x = x

e_tr = tr_inds.numpy()

e_val = np.concatenate([tr_inds.numpy(), val_inds.numpy()])

tr_edge_index = edge_index[:, e_tr]
tr_edge_attr = edge_attr[e_tr]
tr_y = y[e_tr]
tr_edge_times = timestamps[e_tr]

val_edge_index = edge_index[:, e_val]
val_edge_attr = edge_attr[e_val]
val_y = y[e_val]
val_edge_times = timestamps[e_val]

te_edge_index = edge_index
te_edge_attr = edge_attr
te_y = y
te_edge_times = timestamps

tr_data = Data(x = tr_x,edge_index = tr_edge_index, edge_attr = tr_edge_attr, y = tr_y)
val_data = Data(x = val_x,edge_index = val_edge_index, edge_attr = val_edge_attr,y = val_y)
te_data = Data(x = te_x,edge_index = te_edge_index, edge_attr = te_edge_attr,  y = te_y)

tr_data.timestamps = tr_edge_times
val_data.timestamps = val_edge_times
te_data.timestamps = te_edge_times

print('Train Data :', tr_data)
print('Val Data :', val_data)
print('Test Data :', te_data)

6. Z-Score Normalize Node and Edge Features

Normalizes all feature columns to zero mean and unit variance.
This prevents features with large numeric ranges (like Amount Received in thousands)
from dominating features with small numeric ranges (like currency/format integers).
z_norm is computed on train data and applied to all three splits to prevent data leakage.

In [10]:
def z_norm(data):
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1, dtype=torch.float), std)
    return (data - data.mean(0).unsqueeze(0)) / std

tr_data.x = val_data.x = te_data.x = z_norm(tr_data.x)

tr_data.edge_attr = z_norm(tr_data.edge_attr)
val_data.edge_attr = z_norm(val_data.edge_attr)
te_data.edge_attr = z_norm(te_data.edge_attr)

print('Edge attr after z-norm (train, row 0) :', tr_data.edge_attr[0].tolist())
print('Edge attr mean (should be ~0) :', tr_data.edge_attr.mean(0).tolist())
print('Edge attr std (should be ~1) :', tr_data.edge_attr.std(0).tolist())

7. Final Summary

In [11]:
print(f'Train -> {tr_data.num_edges} edges, {tr_inds.shape[0]/total*100:.1f}% of transactions')
print(f'Val -> {val_data.num_edges:,} edges, cumulative (train + val days)')
print(f'Test -> {te_data.num_edges:,} edges, full graph (all days)')

8. Save Split Data Objects for GNN Training

In [12]:
import os
local_dir = "/home/shreyas-nalle/Desktop/Delusional/model"
save_path = os.path.join(local_dir, "split_data.pt") if os.path.exists(local_dir) else "split_data.pt"
torch.save({
    "tr_data"  : tr_data,
    "val_data" : val_data,
    "te_data"  : te_data,
    "tr_inds"  : tr_inds,
    "val_inds" : val_inds,
    "te_inds"  : te_inds
}, save_path)
print(f"Saved split_data.pt -> {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1e9:.2f} GB")